In [2]:
!git clone https://github.com/nshoa/llm-learning.git

Cloning into 'llm-learning'...
remote: Enumerating objects: 177, done.
remote: Counting objects: 100% (177/177), done.
remote: Compressing objects: 100% (134/134), done.
remote: Total 177 (delta 61), reused 146 (delta 33), pack-reused 0 (from 0)
Receiving objects: 100% (177/177), 260.99 KiB | 16.31 MiB/s, done.
Resolving deltas: 100% (61/61), done.


In [7]:
%cd llm-learning/courses/deeplearning_ai_rag/module_01_rag_overview/llm_calls_and_simple_augmented_prompts

/content/llm-learning/courses/deeplearning_ai_rag/module_01_rag_overview/llm_calls_and_simple_augmented_prompts


In [10]:
!pip install together

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 330.8/330.8 kB 28.3 MB/s eta 0:00:00


In [8]:
from utils import (
    generate_with_single_input,
    generate_with_multiple_input,
    get_proxy_url,
    get_proxy_headers,
    get_together_key
)

In [2]:
from dotenv import load_dotenv
load_dotenv()

True

In [6]:
output = generate_with_single_input(
    prompt="What is the capital of Vietnam?"
)

print(output)

print("Role:", output['role'])
print("Content:", output['content'])

{'role': <MessageRole.ASSISTANT: 'assistant'>, 'content': 'The capital of Vietnam is Hanoi.'}
Role: MessageRole.ASSISTANT
Content: The capital of Vietnam is Hanoi.


In [9]:
messages = [
    {'role': 'user', 'content': 'Hello, who won the FIFA world cup in 2018?'},
    {'role': 'assistant', 'content': 'France won the 2018 FIFA World Cup.'},
    {'role': 'user', 'content': 'Who was the captain?'}
]

output = generate_with_multiple_input(
    messages=messages,
    max_tokens=100
)

print("Role:", output['role'])
print("Content:", output['content'])

Role: MessageRole.ASSISTANT
Content: The captain of the French team that won the 2018 FIFA World Cup was Hugo Lloris.


In [10]:
from openai import OpenAI, DefaultHttpxClient
import httpx

In [11]:
base_url = get_proxy_url() # If using together endpoint, add it here https://api.together.xyz/

# Custom transport to bypass SSL verification. This is only needed if using our proxy. Otherwise you can ignore it.
transport = httpx.HTTPTransport(local_address="0.0.0.0", verify=False)

# Create a DefaultHttpxClient instance with the custom transport
http_client = DefaultHttpxClient(transport=transport, headers=get_proxy_headers())

client = OpenAI(
    api_key = get_together_key(), # Set any as our proxy does not use it. Set the together api key if using the together endpoint.
    base_url=base_url,
    http_client=http_client, # ssl bypass to make it work via proxy calls, remove it if running with together.ai endpoint
)

api key: f9d7fa9747481cbf3601c9f74b0c194cf0f564b518a194a7ce16c74fc1a48580


In [12]:
messages = [
    {'role': 'user', 'content': 'Hello, who won the FIFA world cup in 2018?'},
    {'role': 'assistant', 'content': 'France won the 2018 FIFA World Cup.'},
    {'role': 'user', 'content': 'Who was the captain?'}
]

In [13]:
response = client.chat.completions.create(messages = messages, model ="meta-llama/Llama-3.2-3B-Instruct-Turbo")

In [14]:
print(response)

ChatCompletion(id='oYF6Wpk-4msxKE-9d153c25284d1fb8', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='The captain of the French team that won the 2018 FIFA World Cup was Hugo Lloris.', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[]), seed=14870866794133770000)], created=1771666423, model='meta-llama/Llama-3.2-3B-Instruct-Turbo', object='chat.completion', service_tier=None, system_fingerprint=None, usage=CompletionUsage(completion_tokens=22, prompt_tokens=73, total_tokens=95, completion_tokens_details=None, prompt_tokens_details=None, cached_tokens=0), prompt=[])


In [16]:
house_data = [
    {
        "address": "123 Maple Street",
        "city": "Springfield",
        "state": "IL",
        "zip": "62701",
        "bedrooms": 3,
        "bathrooms": 2,
        "square_feet": 1500,
        "price": 230000,
        "year_built": 1998
    },
    {
        "address": "456 Elm Avenue",
        "city": "Shelbyville",
        "state": "TN",
        "zip": "37160",
        "bedrooms": 4,
        "bathrooms": 3,
        "square_feet": 2500,
        "price": 320000,
        "year_built": 2005
    }
]

In [18]:
# First, let's create a layout for the houses

def house_info_layout(houses):
    # Create an empty string
    layout = ''
    # Iterate over the houses
    for house in houses:
        # For each house, append the information to the string using f-strings
        # The following way using brackets is a good way to make the code readable as in each line you can start a new f-string that will appended to the previous one
        layout += (f"House located at {house['address']}, {house['city']}, {house['state']} {house['zip']} with "
            f"{house['bedrooms']} bedrooms, {house['bathrooms']} bathrooms, "
            f"{house['square_feet']} sq ft area, priced at ${house['price']}, "
            f"built in {house['year_built']}.\n") # Don't forget the new line character at the end!
    return layout

In [19]:
# Check the layout
print(house_info_layout(house_data))

House located at 123 Maple Street, Springfield, IL 62701 with 3 bedrooms, 2 bathrooms, 1500 sq ft area, priced at $230000, built in 1998.
House located at 456 Elm Avenue, Shelbyville, TN 37160 with 4 bedrooms, 3 bathrooms, 2500 sq ft area, priced at $320000, built in 2005.



In [20]:
def generate_prompt(query, houses):
    # The code made above is modular enough to accept any list of houses, so you could also choose a subset of the dataset.
    # This might be useful in a more complex context where you want to give only some information to the LLM and not the entire data
    houses_layout = house_info_layout(houses)
    # Create a hard-coded prompt. You can use three double quotes (") in this cases, so you don't need to worry too much about using single or double quotes and breaking the code
    PROMPT = f"""
Use the following houses information to answer users queries.
{houses_layout}
Query: {query}
             """
    return PROMPT

In [21]:
print(generate_prompt("What is the most expensive house?", houses = house_data))


Use the following houses information to answer users queries.
House located at 123 Maple Street, Springfield, IL 62701 with 3 bedrooms, 2 bathrooms, 1500 sq ft area, priced at $230000, built in 1998.
House located at 456 Elm Avenue, Shelbyville, TN 37160 with 4 bedrooms, 3 bathrooms, 2500 sq ft area, priced at $320000, built in 2005.

Query: What is the most expensive house?
             


In [22]:
query = "What is the most expensive house? And the bigger one?"
# Asking without the augmented prompt, let's pass the role as user
query_without_house_info = generate_with_single_input(prompt = query, role = 'user')
# With house info, given the prompt structuer, let's pass the role as assistant
enhanced_query = generate_prompt(query, houses = house_data)
query_with_house_info = generate_with_single_input(prompt = enhanced_query, role = 'assistant')

In [23]:
print(query_without_house_info['content'])

The most expensive house and the largest house in the world are often debated topics, as there are various sources and definitions of "most expensive" and "largest." However, here are some of the most notable examples:

**Most Expensive House:**

The most expensive house in the world is the Antilia building, which is the private residence of Indian business magnate Mukesh Ambani and his family. Located in Mumbai, India, Antilia is valued at over $1 billion USD. The building was designed by the Chicago-based architectural firm Perkins+Will and took over 600 workers six years to complete. It has 27 floors, 168 car garage, 50-seat movie theater, 3 helipads, and a health spa.

**Largest House:**

The largest house in the world is the Biltmore Estate, located in Asheville, North Carolina, USA. It was built by George Vanderbilt II in the late 1800s and covers an area of 175,000 square feet (16,300 square meters). The estate has 250 rooms, including 35 bedrooms, 43 bathrooms, and 65 fireplace

In [24]:
print(query_with_house_info['content'])

The most expensive house is the one located at 456 Elm Avenue, Shelbyville, TN 37160, priced at $320,000.

The bigger house is the one located at 456 Elm Avenue, Shelbyville, TN 37160, with 4 bedrooms, 3 bathrooms, and 2500 sq ft area.
